# Optional: Can you trust the instrument's spectrum? Process the raw data yourself

You'll turn the raw signal the instrument records into a spectrum and compare it with the
instrument's own.

The spectrometer records a **free induction decay** (FID): signal against time. Here you process
the Ala-Ala FIDs from notebook 01 into spectra yourself, one step per cell.

Time: about 30 minutes. Do notebook 01 first. Boxes headed **RECORD YOUR ANSWER** are where you
write something; being wrong is fine, since the reveal shows what happened.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0. Paper: *Scientific Reports* 14, 17083 (2024).

In [ ]:
# Setup. In Colab this first installs the training package and the Rakiura helper (about 30 seconds).
import importlib.util, os
if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"
import matplotlib.pyplot as plt
import nmrtrain as nt

NOTEBOOK = "notebooks/04_process_raw_fids.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

In [ ]:
paths = nt.data.fetch("11223995", "AA", acid="K", nucleus="1H", times=["1day", "4Month"])
dic, fid, info = nt.io.load_fid(paths[0])
print(f"{info.label}: {fid.size} complex points, sweep width {dic['acqus']['SW_h']:.0f} Hz")

fig_fid, ax = plt.subplots(figsize=(10, 3))
ax.plot(fid.real[:4000], lw=0.6)
ax.set_xlabel("time (points)"); ax.set_yticks([]); ax.set_title("the raw FID (first 4000 points)")
plt.show()

### What you need

The instrument made its own spectrum from this same raw signal, with the same steps you are about
to run, so a careful run of your own should match it.

The Fourier transform gives a Bruker spectrum mirrored: high and low ppm swap ends of the axis.
In 98% acid the acid peak sits near 11.5 ppm. Mirroring flips the axis end to end around its middle,
about 6 ppm on this scale.

Longer version: NMR project step 3, Can you trust the instrument's spectrum? Process the raw data yourself.

> **RECORD YOUR ANSWER: predict before you process anything**
>
> **What gets saved:** `MY_PREDICTION`, two short answers, saved with the time.
>
> **Where to look:** the paragraph above, and the acid peak you found in notebook 01.
>
> **How to fill this in:**
> - `matches_instrument`: will your spectrum match the instrument's: `"yes"`, `"no"` or `"unsure"`
> - `acid_ppm_if_not_reversed`: where the acid peak lands, in ppm, if you skip reversing the axis (step 6 below), a number like `3.0`

<details><summary>Stuck? Open hint 1</summary>

How far is 11.5 ppm from the middle of the axis at about 6 ppm? Go the same distance on the other side.

</details>

<details><summary>Stuck? Open hint 2</summary>

11.5 is about 5.5 above 6, so the mirror image lands about 5.5 below 6.

</details>

In [ ]:
MY_PREDICTION = {
    "matches_instrument": ...,         # "yes", "no" or "unsure"
    "acid_ppm_if_not_reversed": ...,   # a number in ppm: where the acid peak lands if you skip the reverse step
}
nt.record.predict("raw", MY_PREDICTION)

## The steps

The next two cells run steps 1 to 5:

1. Remove the digital filter delay (a Bruker detail).
2. Line broadening: light smoothing. The instrument used 0.5 Hz.
3. Zero-fill: pad with zeros for a finer ppm grid.
4. Fourier transform: time to frequency.
5. Phase: make the peaks point up and stand straight (automatic here).

In [ ]:
f1 = nt.process.remove_digital_filter(dic, fid)
f2 = nt.process.line_broaden(dic, f1, lb_hz=0.5)

### What you need

Zero-filling pads the recording with zeros before the transform. That gives more points per peak
(a finer ppm grid), so peaks come out smoother. It adds no new information.

Longer version: NMR project step 3, Can you trust the instrument's spectrum? Process the raw data yourself.

> **RECORD YOUR ANSWER: choose the zero-fill**
>
> **What gets saved:** `ZERO_FILL`, the number of points after padding.
>
> **Where to look:** the overlay plots in the reveal below, after you run it.
>
> **How to fill this in:**
> - Keep `65536` for the first run. Then try `8192`, run the cells down to the reveal again, and compare the peak shapes.

<details><summary>Stuck? Open hint 1</summary>

Run once with each value and look at the tops of the peaks in the reveal plots.

</details>

<details><summary>Stuck? Open hint 2</summary>

With fewer points, peaks look more jagged and their tops can fall between points. 65536 is the
finer grid.

</details>

In [ ]:
ZERO_FILL = 65536   # the number of points after padding: 65536, or try 8192
nt.record.saved("ZERO_FILL", ZERO_FILL, "the number of points after zero-filling")
f3 = nt.process.zero_fill(f2, ZERO_FILL)
spec = nt.process.fft(f3)
spec = nt.process.autophase(spec)

### Trap: the mirrored axis

The next cell checks your spectrum before the last steps. Compare where the acid peak lands with
your prediction.

In [ ]:
ppm = nt.process.ppm_axis(dic, spec)
not_reversed = info.with_data(ppm, spec.real, processing="raw")
print(nt.checks.check_axis(not_reversed))

The next cell runs the last two steps and checks the axis again:

6. Reverse: Bruker spectra come out of the transform mirrored.
7. Baseline: subtract a smooth curve so the floor sits at zero.

In [ ]:
mine = info.with_data(ppm, nt.process.baseline(nt.process.reverse(spec).real), processing="raw")
print(nt.checks.check_axis(mine))

## Reveal: your spectrum over the instrument's

The next cell processes the 1 day and 4 month FIDs your way and lays each over the instrument's
spectrum, with a similarity value for each.

In [ ]:
fig_overlay, sims = None, {}
if nt.record.revealed("raw"):
    fig_overlay, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, path in zip(axes, paths):
        inst = nt.load_processed(path)
        own = nt.process.process_fid(path, size=ZERO_FILL)
        sims[inst.time] = nt.peaks.similarity(own, inst)
        nt.plot.stack([inst, own], 1.0, 6.5, norm=(1.0, 6.5), offset=0.0, ax=ax,
                      labels=["instrument", "mine"], title=inst.label)
        print(nt.checks.similarity_feedback(sims[inst.time]))
    plt.show()

A similarity near 1 means your processing reproduces the instrument's. Small leftovers are baseline
and phase detail; they do not move any peak.

> **SAVE YOUR HAND-IN: your processing hand-in**
>
> **What gets saved:** in `handin/`: the overlay and `verdict.json` (your prediction, the similarities and your zero-fill).
>
> **Where to look:** nothing new to fill in: the cell uses what you ran above.
>
> **How to fill this in:**
> - Run the cell. Nothing is sent for review until you run `nt.portal.submit()`.

<details><summary>Stuck? Open hint 1</summary>

If it saves no figure, run the reveal cell above first, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin("raw-overlay", figures={"overlay": fig_overlay} if fig_overlay else {},
                          verdict={"prediction": MY_PREDICTION, "similarity": sims, "zero_fill": ZERO_FILL},
                          numbers={"min_similarity": round(min(sims.values()), 3) if sims else "blank"},
                          question="raw", processing="raw")
# nt.portal.submit()